# Langchain tools and function calling

### Introduction

In this section, we will explore how to use the LLM model deployed in the previous section with `langchain` and how to extend its capabilities by enabling it to call external tools and functions.

### Prerequisites

- Completion of steps in `010_langchain_fundamentals` where we deployed an Azure Foundry with an LLM model.

### Getting the LLM Endpoint and API Key

To use the LLM model with `langchain`, we first need to retrieve the endpoint and API key from the Terraform outputs. You can do this by running the following command.

In [1]:
foundry_endpoint = ! terraform -chdir=infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name = ! terraform -chdir=infra output -raw llm_model_deployment_name
llm_model_deployment_name = llm_model_deployment_name.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name)

Foundry Endpoint: https://foundry-400.cognitiveservices.azure.com/
Foundry API Key: AAACOGYQ3t...
LLM Model Deployment Name (ChatGPT): gpt-5.4


### 1. Simple Chat - No Tools

`OpenAI` API defined the standard spec for calling LLM models. Most of the inference tools and frameworks are built on top of this API providing a kind of wrapper. LangChain is one of them.

Create a `ChatOpenAI` model pointing at the Azure Foundry OpenAI-compatible endpoint and send a basic message. Note how we can enable streaming responses for real-time output. And note also how much similar the code is to calling OpenAI's API directly.

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name,
    streaming=True,
    max_completion_tokens= 512
)

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I’m ChatGPT, an AI assistant created by OpenAI.

I can help with things like:
- answering questions
- explaining concepts
- writing and editing
- brainstorming ideas
- summarizing information
- coding help
- planning, research, and problem-solving

A few useful things to know about me:
- I don’t have feelings, beliefs, or personal experiences.
- I generate responses based on patterns in data and the context of our conversation.
- I can be very helpful, but I can also be wrong, so it’s good to double-check important information.
- I don’t automatically know private or real-time information unless you provide it or my environment explicitly gives me access to it.

You can talk to me casually, ask for short or detailed answers, or have me adapt to a style you prefer.

If you want, I can also tell you:
- what I’m good at
- what my limitations are
- how to get better results from me

### 2. Define Tools

Create custom Python functions as tools using the `@tool` decorator. These will be available for the agent to call when needed.

In [4]:
from langchain_core.tools import tool
from random import randint


@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    temp = randint(10, 30)
    return f"The weather in {location} is {conditions[randint(0, 3)]} with a high of {temp}°C."


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together.

    Args:
        a: first number
        b: second number
    """
    return a * b


tools = [get_weather, multiply]
print("Registered tools:", [t.name for t in tools])

Registered tools: ['get_weather', 'multiply']


### 3. Tool Binding — Test Tool Calling

Bind the tools to the model and verify the LLM can decide when to call them.

In [5]:
model_with_tools = model.bind_tools(tools)

# This should NOT trigger a tool call
response = model_with_tools.invoke([HumanMessage(content="Hi there!")])
print(f"Content: {response.content}")
print(f"Tool calls: {response.tool_calls}")

Content: Hi! How can I help?
Tool calls: []


In [6]:
# This SHOULD trigger a tool call
response = model_with_tools.invoke([HumanMessage(content="What's the weather in Amsterdam?")])
print(f"Content: {response.content}")
print(f"Tool calls: {response.tool_calls}")

Content: 
Tool calls: [{'name': 'get_weather', 'args': {'location': 'Amsterdam'}, 'id': 'call_NMWq3Y2yCjdLUnAnpqhrRVwu', 'type': 'tool_call'}]


### 4. Create a ReAct Agent

Use LangGraph's `create_react_agent` to build an agent that can reason about when to call tools, execute them, and incorporate results into its response.

The agent implements the **ReAct** (Reasoning + Acting) pattern: it thinks about what to do, calls a tool if needed, observes the result, and repeats until it has a final answer.

In [7]:
from langchain.agents import create_agent

agent = create_agent(model, tools)

### 5. Run the Agent

#### No tool needed — simple question

In [14]:
async for step in agent.astream(
    {"messages": [HumanMessage(content="Tell me about yourself")]},
    stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me about yourself
================================== Ai Message ==================================

I’m an AI assistant that helps with writing, explaining, brainstorming, coding, math, planning, and general questions.

A few useful things about me:
- I can explain topics simply or in depth.
- I can help draft emails, essays, summaries, and ideas.
- I can assist with programming and debugging.
- I can do reasoning and basic problem-solving.
- I can use available tools, when needed, for things like checking weather or doing calculations.

A few limitations:
- I can be mistaken, so it’s good to verify important facts.
- I don’t have personal experiences or feelings.
- I don’t automatically know private or real-time information unless a tool provides it.

If you want, I can also tell you:
- what I’m good at,
- how to use me effectively,
- or give a short/fun version of this intro.


### Tool call — weather query

The agent should recognize this requires the `get_weather` tool, call it, then respond with the result.

In [15]:
async for step in agent.astream(
    {"messages": [HumanMessage(content="What's the weather like in Amsterdam?")]},
    stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What's the weather like in Amsterdam?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_Zp7ANF2Zd6ZIFp53tbUUunn9)
 Call ID: call_Zp7ANF2Zd6ZIFp53tbUUunn9
  Args:
    location: Amsterdam
================================= Tool Message =================================
Name: get_weather

The weather in Amsterdam is stormy with a high of 22°C.
================================== Ai Message ==================================

It’s stormy in Amsterdam with a high of 22°C.


### Tool call — multiply

In [16]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is 7 multiplied by 13?")]}
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What is 7 multiplied by 13?
================================== Ai Message ==================================
Tool Calls:
  multiply (call_Qxux22f963VIL8mPII9TxM4S)
 Call ID: call_Qxux22f963VIL8mPII9TxM4S
  Args:
    a: 7
    b: 13
================================= Tool Message =================================
Name: multiply

91
================================== Ai Message ==================================

91


### More Resources

- [LangChain Tool Calling](https://python.langchain.com/docs/concepts/tool_calling/)
- [LangGraph ReAct Agent](https://python.langchain.com/docs/tutorials/agents/)
- [ChatOpenAI with custom endpoints](https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.base.ChatOpenAI.html)
- [LangChain MCP Adapters](https://github.com/langchain-ai/langchain-mcp-adapters) — connect LangChain agents to MCP servers
- [Model Context Protocol (MCP)](https://modelcontextprotocol.io/introduction) — the open standard for tool interoperability
- [Microsoft Learn MCP Server](https://learn.microsoft.com/api/mcp) — search Microsoft documentation via MCP